# MicroDuck RL — Colab A100 Trainer

공식 `pollen-robotics/microduck_rl`의 최신 `develop` 브랜치를 사용해 MicroDuck walking policy를 학습합니다.

- A100 GPU
- 4096 parallel environments
- 기본 목표 6000 PPO iterations
- upstream 기본 checkpoint 간격 250 iterations
- Google Drive 자동 backup
- Colab 세션 종료 후 자동 resume
- 최초 실행 시 64 env / 5 iteration smoke test

먼저 **Runtime → Change runtime type → A100 GPU**를 선택하세요.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess

print("GPU")
subprocess.run([
    "nvidia-smi",
    "--query-gpu=name,memory.total,driver_version",
    "--format=csv,noheader",
], check=False)

## 1. 학습 설정

처음에는 기본값 그대로 실행하는 것을 권장합니다.

In [ ]:
TASK_ID = "Mjlab-Velocity-Flat-MicroDuck"
EXPERIMENT_NAME = "velocity"

NUM_ENVS = 4096
TARGET_ITERS = 6000
SYNC_INTERVAL = 60

DRIVE_ROOT = "/content/drive/MyDrive/microduck-training"
SMOKE_TEST = True

print({
    "TASK_ID": TASK_ID,
    "EXPERIMENT_NAME": EXPERIMENT_NAME,
    "NUM_ENVS": NUM_ENVS,
    "TARGET_ITERS": TARGET_ITERS,
    "SYNC_INTERVAL": SYNC_INTERVAL,
    "DRIVE_ROOT": DRIVE_ROOT,
    "SMOKE_TEST": SMOKE_TEST,
})

## 2. 공식 MicroDuck RL 설치

이 저장소는 Colab 실행 도우미만 가지고 있습니다.
실제 RL 코드는 `/content/microduck_rl`에 공식 upstream을 clone합니다.

In [ ]:
import os
import subprocess

WRAPPER_DIR = "/content/microduck_colab"

if os.path.isdir(os.path.join(WRAPPER_DIR, ".git")):
    subprocess.run(["git", "-C", WRAPPER_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run([
        "git", "clone",
        "https://github.com/namyikim/microduck.git",
        WRAPPER_DIR,
    ], check=True)

subprocess.run(
    ["bash", f"{WRAPPER_DIR}/scripts/setup_colab.sh"],
    check=True,
)

## 3. 학습 시작 / 자동 resume

이 셀은 Drive에 이전 checkpoint가 있으면 자동으로 복원해서 이어서 학습합니다.

학습 중에는 약 60초마다 Drive에 백업합니다.
Colab 세션이 종료되면 새 세션에서 이 노트북을 다시 위에서부터 실행하면 됩니다.

In [ ]:
import os
import subprocess

env = os.environ.copy()
env.update({
    "TASK_ID": TASK_ID,
    "EXPERIMENT_NAME": EXPERIMENT_NAME,
    "NUM_ENVS": str(NUM_ENVS),
    "TARGET_ITERS": str(TARGET_ITERS),
    "SYNC_INTERVAL": str(SYNC_INTERVAL),
    "DRIVE_ROOT": DRIVE_ROOT,
    "SMOKE_TEST": "1" if SMOKE_TEST else "0",
    "WANDB_MODE": env.get("WANDB_MODE", "offline"),
})

subprocess.run(
    ["bash", f"{WRAPPER_DIR}/scripts/train_colab.sh"],
    env=env,
    check=True,
)

## 4. Drive checkpoint 확인

In [ ]:
from pathlib import Path
import re

root = Path(DRIVE_ROOT) / "logs" / "rsl_rl" / EXPERIMENT_NAME
items = []

if root.exists():
    for p in root.rglob("model_*.pt"):
        m = re.fullmatch(r"model_(\d+)\.pt", p.name)
        if m:
            items.append((int(m.group(1)), p))

items.sort(key=lambda x: x[0])

if not items:
    print("아직 checkpoint가 없습니다.")
else:
    print("checkpoint count:", len(items))
    for n, p in items[-10:]:
        print(f"{n + 1:6d} iterations (approx.)  {p}")
    print("\nLatest:", items[-1][1])

## 5. 학습 완료 후

학습 완료 후에는 official upstream의 `play`, `scripts/export.py`를 사용해
simulation 검증과 ONNX export를 진행할 수 있습니다.

실제 로봇에 배포하기 전 simulation에서 policy를 먼저 확인하세요.